# E-Commerce Supply Chain & Delivery Performance Analysis

## 1. Introduction & Setup
This notebook processes raw supply chain data to identify operational bottlenecks, calculate transit times, and prepare the dataset for SQL querying and Power BI visualization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_theme(style='whitegrid')

## 2. Data Loading & Inspection

In [ ]:
df = pd.read_csv('ecommerce_supply_chain_data.csv')

# Convert date columns to datetime objects
date_cols = ['order_date', 'ship_date', 'delivery_date']
for col in date_cols:
    df[col] = pd.to_datetime(df[col])

display(df.head())
display(df.info())

## 3. Data Cleaning & Handling Missing Values

In [ ]:
# Check for missing values
print(df.isnull().sum())

# Impute missing shipping costs with median cost per carrier
df['shipping_cost'] = df['shipping_cost'].fillna(df.groupby('carrier')['shipping_cost'].transform('median'))

# Verify lost packages have no delivery date
print("Lost packages missing delivery dates:", df[df['delivery_status'] == 'Lost']['delivery_date'].isnull().all())

## 4. Feature Engineering
We need to calculate internal processing time and external transit time to isolate where delays occur.

In [ ]:
df['processing_time_days'] = (df['ship_date'] - df['order_date']).dt.days
df['actual_transit_time_days'] = (df['delivery_date'] - df['ship_date']).dt.days

# Flag SLA Breaches (Assume SLA is 5 days max transit)
df['sla_breach'] = df['actual_transit_time_days'] > 5

df.head()

## 5. Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(10,6))
sns.countplot(data=df, x='carrier', hue='delivery_status')
plt.title('Delivery Status by Carrier')
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
sns.boxplot(data=df, x='warehouse_id', y='processing_time_days')
plt.title('Warehouse Processing Times')
plt.show()

## 6. Export Cleaned Data for SQL / BI

In [ ]:
df.to_csv('cleaned_supply_chain_data.csv', index=False)
print('Cleaned dataset saved successfully.')